# 第8周：Pseudobulk + edgeR 年龄组单模型

本 Notebook **只使用一个设计模型**：

```r
~ age_group + sex + technology
```

比较 **Old vs Young**，同时控制 sex 和 technology。

流程：
1. 读取 pseudobulk counts / metadata
2. 排除 Week 7 离群样本
3. 强制排除 `30-M-2_Lung_fibroblast/stromal`
4. 只保留 Young / Old
5. 使用 `edgeR::filterByExpr()`
6. TMM normalization
7. edgeR QL GLM
8. 检验 `age_groupold`
9. FDR < 0.05 且 |logFC| >= 1
10. 输出 DE 表、汇总表和火山图

**不进行连续年龄 `age_months` 模型。**


In [ ]:
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

print("Rscript:", shutil.which("Rscript"))
if shutil.which("Rscript") is None:
    raise EnvironmentError("找不到 Rscript，请检查 R 是否已安装并加入 PATH。")

COUNT_FILE = Path("../data_processed/pseudobulk_counts.tsv")
META_FILE = Path("../data_processed/pseudobulk_metadata.tsv")
OUTLIER_FILE = Path("../results/pseudobulk/sample_outliers.csv")

OUT_DIR = Path("../results/de")
FIG_DIR = Path("../figures/de")
TMP_DIR = OUT_DIR / "tmp_edgeR"

for d in [OUT_DIR, FIG_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for f in [COUNT_FILE, META_FILE, OUTLIER_FILE]:
    print(f"{f}: {'✓ 存在' if f.exists() else '✗ 不存在'}")

if not all(f.exists() for f in [COUNT_FILE, META_FILE, OUTLIER_FILE]):
    raise FileNotFoundError("请检查上面的文件路径。")

Rscript: C:\Program Files\R\R-4.6.1\bin\x64\Rscript.EXE
..\data_processed\pseudobulk_counts.tsv: ✓ 存在
..\data_processed\pseudobulk_metadata.tsv: ✓ 存在
..\results\pseudobulk\sample_outliers.csv: ✓ 存在


In [19]:
counts = pd.read_csv(COUNT_FILE, sep="\t", index_col=0)
meta = pd.read_csv(META_FILE, sep="\t", index_col=0)
outlier_df = pd.read_csv(OUTLIER_FILE)

common = counts.columns.intersection(meta.index)
if len(common) == 0:
    raise ValueError("counts 和 metadata 没有共同 sample ID。")

counts = counts.loc[:, common]
meta = meta.loc[common].copy()

print("counts:", counts.shape)
print("metadata:", meta.shape)
print("metadata columns:", meta.columns.tolist())

counts: (3000, 110)
metadata: (110, 6)
metadata columns: ['mouse.id', 'tissue', 'major_cell_type', 'age_months', 'sex', 'cell_index']


## ④ 检查 metadata

必须存在：

- `tissue`
- `major_cell_type`
- `age_group`

如果没有 `age_group`，但存在 `age_months`，则自动生成：

- <= 3：young
- > 24：old
- 中间：middle

随后只保留 young 和 old。

In [ ]:
required = ["tissue", "major_cell_type"]

missing = [x for x in required if x not in meta.columns]
if missing:
    raise KeyError("metadata 缺少必要字段: " + ", ".join(missing))

if "age_group" not in meta.columns:
    if "age_months" not in meta.columns:
        raise KeyError("metadata 同时缺少 age_group 和 age_months。")

    meta["age_months"] = pd.to_numeric(meta["age_months"], errors="coerce")

    if meta["age_months"].isna().any():
        raise ValueError("存在无法解析的 age_months。")

    meta["age_group"] = pd.cut(
        meta["age_months"],
        bins=[-np.inf, 3, 24, np.inf],
        labels=["young", "middle", "old"],
        right=True
    )
else:
    meta["age_group"] = (meta["age_group"].astype(str).str.lower().str.strip())

print(meta["age_group"].value_counts(dropna=False))

print("sex:","存在" if "sex" in meta.columns else "不存在，将不加入模型")
print("technology:","存在" if "technology" in meta.columns else "不存在，将不加入模型")

age_group
middle    52
young     32
old       26
Name: count, dtype: int64
sex: 存在
technology: 不存在，将不加入模型


## ⑤ 排除离群样本

排除：

1. Week 7 `sample_outliers.csv` 中已经标记的样本
2. 强制排除 `30-M-2_Lung_fibroblast/stromal`

随后只保留 Young / Old。

In [ ]:
outlier_ids = set(outlier_df.iloc[:, 0].dropna().astype(str))

matched_outliers = [
s for s in counts.columns
    if s in outlier_ids
]

forced = "30-M-2_Lung_fibroblast/stromal"

if forced in counts.columns and forced not in matched_outliers:
    matched_outliers.append(forced)

matched_outliers = list(dict.fromkeys(matched_outliers))

print("最终排除样本数:", len(matched_outliers))
for s in matched_outliers:
    print(" -", s)

excluded = meta.loc[[s for s in matched_outliers if s in meta.index]].copy()

if len(excluded):
    excluded.insert(0, "sample_id", excluded.index)
    excluded["outlier_reason"] = "Week 7 outlier / forced exclusion"

excluded.to_csv(OUT_DIR / "sample_exclusion_audit.tsv",sep="\t",index=False)

counts = counts.drop(columns=matched_outliers, errors="ignore")
meta = meta.drop(index=matched_outliers, errors="ignore")

keep_age = meta["age_group"].isin(["young", "old"])
counts = counts.loc[:, keep_age]
meta = meta.loc[keep_age].copy()

counts.to_csv(TMP_DIR / "counts.tsv", sep="\t")
meta.to_csv(TMP_DIR / "metadata.tsv", sep="\t")

print("\nYoung / Old 样本数:")
print(meta["age_group"].value_counts())

if forced in counts.columns:
    raise RuntimeError("30-M-2 仍然存在。")

最终排除样本数: 6
 - 1-M-62_Liver_fibroblast/stromal
 - 18-F-51_Liver_fibroblast/stromal
 - 21-F-54_Liver_fibroblast/stromal
 - 24-M-58_Liver_endothelial
 - 3-F-56_Liver_fibroblast/stromal
 - 30-M-2_Lung_fibroblast/stromal

Young / Old 样本数:
age_group
young     30
old       25
middle     0
Name: count, dtype: int64


## ⑥ 生成 edgeR 单模型脚本

固定目标设计：

```r
~ age_group + sex + technology
```

目标系数：

```r
age_groupold
```

如果 sex / technology 不存在或只有一个水平，则自动省略。

如果设计矩阵不满秩，则优先尝试去掉 technology，再尝试去掉 sex。

In [35]:
def rpath(p):
    return str(p.resolve()).replace("\\", "/").replace('"', '\\"')

r_script = r"""
suppressPackageStartupMessages({
    library(edgeR)
    library(limma)
})

count_file <- "COUNT_FILE"
meta_file <- "META_FILE"
out_dir <- "OUT_DIR"
fig_dir <- "FIG_DIR"

dir.create(out_dir, recursive=TRUE, showWarnings=FALSE)
dir.create(fig_dir, recursive=TRUE, showWarnings=FALSE)

counts <- as.matrix(read.delim(count_file,row.names=1,check.names=FALSE,stringsAsFactors=FALSE))

storage.mode(counts) <- "integer"

meta <- read.delim(meta_file,row.names=1,check.names=FALSE,stringsAsFactors=FALSE)

meta <- meta[colnames(counts), , drop=FALSE]

standard_result <- function(
    tissue,
    cell_type,
    status="SKIP",
    reason=NA_character_,
    formula=NA_character_,
    n_samples=NA_integer_,
    n_young=NA_integer_,
    n_old=NA_integer_,
    residual_df=NA_integer_,
    genes_before=NA_integer_,
    genes_after_filterByExpr=NA_integer_,
    genes_removed=NA_integer_,
    significant_genes=NA_integer_,
    up_genes=NA_integer_,
    down_genes=NA_integer_
) {

    data.frame(
        tissue=as.character(tissue),
        cell_type=as.character(cell_type),
        status=as.character(status),
        reason=as.character(reason),
        formula=as.character(formula),
        n_samples=as.integer(n_samples),
        n_young=as.integer(n_young),
        n_old=as.integer(n_old),
        residual_df=as.integer(residual_df),
        genes_before=as.integer(genes_before),
        genes_after_filterByExpr=as.integer(genes_after_filterByExpr),
        genes_removed=as.integer(genes_removed),
        significant_genes=as.integer(significant_genes),
        up_genes=as.integer(up_genes),
        down_genes=as.integer(down_genes),
        stringsAsFactors=FALSE
    )
}
# ============================================================
# 单个 tissue × cell type
# ============================================================
run_one <- function(tissue_name,celltype_name) {
    keep <- (
        meta$tissue == tissue_name &
        meta$major_cell_type == celltype_name &
        meta$age_group %in% c("young", "old")
    )

    m <- meta[keep,,drop=FALSE]

    cts <- counts[,rownames(m),drop=FALSE]
    # --------------------------------------------------------
    # 样本数检查
    # --------------------------------------------------------
    if(nrow(m) < 4){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="fewer than 4 samples",
                n_samples=nrow(m)
            )
        )
    }

    m$age_group <- factor(m$age_group,levels=c("young","old"))

    ny <- sum(m$age_group == "young")

    no <- sum(m$age_group == "old")

    if(ny < 2 ||no < 2){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="fewer than 2 samples in young or old",
                n_samples=nrow(m),
                n_young=ny,
                n_old=no
            )
        )
    }
    # ========================================================
    # Design:
    # ~ age_group + sex + technology
    # ========================================================
    vars <- "age_group"

    if("sex" %in% colnames(m) &&length(unique(na.omit(m$sex))) >= 2){
        m$sex <- factor(m$sex)
        vars <- c(vars,"sex")
    }

    if("technology" %in% colnames(m) &&length(unique(na.omit(m$technology))) >= 2){
        m$technology <- factor(m$technology)
        vars <- c(vars,"technology")
    }

    make_design <- function(v){
        f <- as.formula(paste("~",paste(v,collapse=" + ")))

        model.matrix(f,data=m)
    }

    design <- make_design(vars)

    # 如果不满秩，先去掉 technology
    if(qr(design)$rank < ncol(design) &&"technology" %in% vars){
        vars <- setdiff(vars,"technology")
        design <- make_design(vars)
    }

    # 如果仍不满秩，再去掉 sex
    if(qr(design)$rank < ncol(design) &&"sex" %in% vars){
        vars <- setdiff(vars,"sex")

        design <- make_design(vars)
    }

    if(qr(design)$rank < ncol(design)){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="design matrix not full rank",
                n_samples=nrow(m),
                n_young=ny,
                n_old=no,
                formula=paste("~",paste(vars,collapse=" + "))
            )
        )
    }

    residual_df <- (nrow(design) -qr(design)$rank)

    if(residual_df < 1){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="insufficient residual degrees of freedom",
                n_samples=nrow(m),
                n_young=ny,
                n_old=no,
                residual_df=residual_df
            )
        )
    }
    # ========================================================
    # edgeR
    # ========================================================
    y <- DGEList(counts=cts)

    keep_gene <- filterByExpr(y,group=m$age_group)

    y <- y[keep_gene,,keep.lib.sizes=FALSE]

    if(nrow(y) < 10){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="fewer than 10 genes after filterByExpr",
                n_samples=nrow(m),
                n_young=ny,
                n_old=no,
                residual_df=residual_df,
                genes_before=nrow(cts),
                genes_after_filterByExpr=nrow(y),
                genes_removed=nrow(cts)-nrow(y)
            )
        )
    }

    # edgeR
    y <- normLibSizes(y,method="TMM")

    y <- estimateDisp(y,design)

    fit <- glmQLFit(y,design,robust=TRUE)

    coef_id <- which(colnames(design) =="age_groupold")

    if(length(coef_id) != 1){
        return(
            standard_result(
                tissue_name,
                celltype_name,
                status="SKIP",
                reason="age_groupold coefficient unavailable"
            )
        )
    }

    qlf <- glmQLFTest(fit,coef=coef_id)

    tab <- topTags(qlf,n=Inf,sort.by="PValue")$table

    tab$gene <- rownames(tab)

    tab$tissue <- tissue_name
    tab$cell_type <- celltype_name
    # ========================================================
    # 显著基因定义
    # ========================================================
    tab$significant <- (tab$FDR < 0.05 &abs(tab$logFC) >= 1)

    tab$direction <- "NS"

    tab$direction[tab$significant &tab$logFC >= 1] <- "Old_higher"

    tab$direction[tab$significant &tab$logFC <= -1] <- "Young_higher"
    # ========================================================
    # 保存 DE 结果
    # ========================================================
    st <- gsub("[^A-Za-z0-9_-]","_",tissue_name)

    ct <- gsub("[^A-Za-z0-9_-]","_",celltype_name)

    write.table(
        tab,
        file.path(out_dir,paste0(st,"_",ct,"_old_vs_young_edgeR.tsv")),
        sep="\t",
        quote=FALSE,
        row.names=FALSE
    )
    # ========================================================
    # 火山图
    #
    # 上调：红色三角形
    # 下调：蓝色正方形
    # 非显著：灰色圆点
    #
    # FDR = 0.05 水平虚线
    # log2FC = -1/+1 垂直虚线
    # ========================================================
    tab$neglog10FDR <- -log10(pmax(tab$FDR,.Machine$double.xmin))

    plot_tab <- tab[is.finite(tab$logFC) &is.finite(tab$neglog10FDR),,drop=FALSE]
    # --------------------------------------------------------
    # 三类点
    # --------------------------------------------------------
    ns <- plot_tab[!(plot_tab$FDR < 0.05 &abs(plot_tab$logFC) >= 1),,drop=FALSE]

    up <- plot_tab[plot_tab$FDR < 0.05 &plot_tab$logFC >= 1,,drop=FALSE]

    down <- plot_tab[plot_tab$FDR < 0.05 &plot_tab$logFC <= -1,,drop=FALSE]
    # --------------------------------------------------------
    # 动态坐标范围
    # --------------------------------------------------------
    x_range <- range(plot_tab$logFC,na.rm=TRUE)

    x_pad <- max(0.5,diff(x_range) * 0.05)

    xlim_use <- c(x_range[1] - x_pad,x_range[2] + x_pad)

    fdr_y <- -log10(0.05)

    y_max <- max(plot_tab$neglog10FDR,fdr_y,na.rm=TRUE)

    ylim_use <- c( 0, y_max * 1.08)
    # --------------------------------------------------------
    # PDF
    # --------------------------------------------------------
    volcano_file <- file.path(fig_dir,paste0(st,"_",ct,"_old_vs_young_volcano.pdf"))

    pdf(volcano_file,width=9,height=7.5)
    # --------------------------------------------------------
    # 1. 非显著：灰色圆点
    # --------------------------------------------------------
    plot(
        ns$logFC,
        ns$neglog10FDR,
        pch=16,
        cex=0.65,
        col="grey75",
        xlim=xlim_use,
        ylim=ylim_use,
        xlab="log2 Fold Change (Old vs Young)",
        ylab=expression(-log[10](FDR)),
        main=paste("Old vs Young:",tissue_name,"|",celltype_name)
    )
    # --------------------------------------------------------
    # 2. 上调：红色三角形
    # --------------------------------------------------------
    if(nrow(up) > 0)
    {points(up$logFC,up$neglog10FDR,pch=24,cex=0.9,col="red3",bg="red3")}
    # --------------------------------------------------------
    # 3. 下调：蓝色正方形
    # --------------------------------------------------------
    if(nrow(down) > 0)
    {points(down$logFC,down$neglog10FDR,pch=22,cex=0.9,col="blue3",bg="blue3")}
    # --------------------------------------------------------
    # 4. FDR = 0.05
    # --------------------------------------------------------
    abline(h=fdr_y,lty=2,lwd=1.2)
    # --------------------------------------------------------
    # 5. log2FC = -1 / +1
    # --------------------------------------------------------
    abline(v=c(-1, 1),lty=2,lwd=1.2)
    # --------------------------------------------------------
    # 6. 标注 Top 10 上调 + Top 10 下调
    # --------------------------------------------------------
    sig <- plot_tab[plot_tab$significant,,drop=FALSE]

    if(nrow(sig) > 0)
    {
        sig_up <- sig[sig$logFC >= 1,,drop=FALSE]

        sig_up <- sig_up[order(sig_up$FDR,-abs(sig_up$logFC)),,drop=FALSE]

        sig_up <- head(sig_up,10)

        sig_down <- sig[sig$logFC <= -1,,drop=FALSE]

        sig_down <- sig_down[order(sig_down$FDR,-abs(sig_down$logFC)),,drop=FALSE]

        sig_down <- head(sig_down,10)

        labels <- rbind(sig_up,sig_down)

        if(nrow(labels) > 0)
        {text(labels$logFC,labels$neglog10FDR,labels$gene,pos=3,cex=0.62,offset=0.35)}
    }
    # --------------------------------------------------------
    # 7. 图例
    # --------------------------------------------------------
    legend(
        "topright",
        legend=c("Not significant","Upregulated","Downregulated"),
        pch=c(16,24,22),
        col=c("grey75","red3","blue3"),
        pt.bg=c("grey75","red3","blue3"),
        pt.cex=c(0.7,0.9,0.9),
        bty="n"
    )
    # --------------------------------------------------------
    # 8. 显著基因数量
    # --------------------------------------------------------
    legend(
        "bottomright",
        legend=c(
            paste0("Up: ",nrow(up)),
            paste0("Down: ",nrow(down)),
            paste0("Significant: ",sum(tab$significant))
        ),
        bty="n",
        cex=0.8
    )

    dev.off()
    # ========================================================
    # 汇总
    # ========================================================
    standard_result(
        tissue_name,
        celltype_name,
        status="OK",
        formula=paste("~",paste(vars,collapse=" + ")),
        n_samples=nrow(m),
        n_young=ny,
        n_old=no,
        residual_df=residual_df,
        genes_before=nrow(cts),
        genes_after_filterByExpr=nrow(y),
        genes_removed=nrow(cts)-nrow(y),
        significant_genes=sum(tab$significant),
        up_genes=nrow(up),
        down_genes=nrow(down)
    )
}
# ============================================================
# 所有 tissue × cell type
# ============================================================
results <- list()
k <- 1

for(t in unique(meta$tissue)){
    for(ct in unique(meta$major_cell_type)){
        results[[k]] <- tryCatch(
            run_one(t,ct),
            error=function(e){
                standard_result(tissue=t,cell_type=ct,status="ERROR",reason=e$message)
            }
        )
        k <- k + 1
    }
}
# ============================================================
# 汇总
# ============================================================
summary_df <- do.call(rbind,results)

write.table(summary_df,file.path(out_dir,"DE_summary_old_vs_young.tsv"),sep="\t",quote=FALSE,row.names=FALSE)

print(summary_df)

cat("\nOld vs Young edgeR analysis completed.\n")
"""

r_script = r_script.replace("COUNT_FILE",rpath(TMP_DIR / "counts.tsv"))

r_script = r_script.replace("META_FILE",rpath(TMP_DIR / "metadata.tsv"))

r_script = r_script.replace("OUT_DIR",rpath(OUT_DIR))

r_script = r_script.replace("FIG_DIR",rpath(FIG_DIR))

r_file = TMP_DIR / "run_edgeR_old_vs_young.R"
r_file.write_text(r_script,encoding="utf-8")

print("R 脚本已生成:", r_file)

R 脚本已生成: ..\results\de_edgeR\tmp_edgeR\run_edgeR_old_vs_young.R


## ⑦ 运行 edgeR

固定检验：

```r
~ age_group + sex + technology
```

目标系数：

```r
age_groupold
```

因此：

- `logFC > 0`：Old 高于 Young
- `logFC < 0`：Young 高于 Old

> **修正版说明：** 不同组织/细胞类型如果因为样本数、设计矩阵或自由度不足而 `SKIP`，现在会返回完全相同的汇总字段，因此不会再出现 `rbind` “变量的列数不正确”。同时新版 edgeR 使用 `normLibSizes(..., method="TMM")`。


In [36]:
res = subprocess.run(["Rscript", str(r_file)],capture_output=True,text=True)

print(res.stdout)

if res.returncode != 0:
    print("\n--- R ERROR ---")
    print(res.stderr)
    raise RuntimeError("edgeR 分析失败。")

print("\n✓ Old vs Young edgeR 分析完成。")

            tissue          cell_type status               reason
1           Kidney        endothelial     OK                 <NA>
2           Kidney fibroblast/stromal     OK                 <NA>
3            Liver        endothelial     OK                 <NA>
4            Liver fibroblast/stromal   SKIP fewer than 4 samples
5             Lung        endothelial     OK                 <NA>
6             Lung fibroblast/stromal     OK                 <NA>
7  Heart_and_Aorta        endothelial     OK                 <NA>
8  Heart_and_Aorta fibroblast/stromal     OK                 <NA>
9              Fat        endothelial   SKIP fewer than 4 samples
10             Fat fibroblast/stromal   SKIP fewer than 4 samples
             formula n_samples n_young n_old residual_df genes_before
1  ~ age_group + sex         9       5     4           6         3000
2  ~ age_group + sex         9       5     4           6         3000
3  ~ age_group + sex         8       5     3           5        

In [ ]:
summary_file = OUT_DIR / "DE_summary_old_vs_young.tsv"

if not summary_file.exists():
    raise FileNotFoundError("没有找到 DE_summary_old_vs_young.tsv")

summary_df = pd.read_csv(summary_file,sep="\t")

display(summary_df)

,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,55.0,23.0,32.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,266.0,111.0,155.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,8,5.0,3.0,5.0,3000.0,416.0,2584.0,12.0,2.0,10.0
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,39.0,28.0,11.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1318.0,1682.0,0.0,0.0,0.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1440.0,1560.0,9.0,4.0,5.0
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
summary_file = OUT_DIR / "DE_summary_old_vs_young.tsv"

if not summary_file.exists():
    raise FileNotFoundError(f"R 没有生成结果文件：{summary_file}\n""请先检查前一个 edgeR Cell 的 R 输出。")

summary_df = pd.read_csv(summary_file,sep="\t")

print("结果文件：", summary_file)
print("结果维度：", summary_df.shape)
print("结果列：")
print(summary_df.columns.tolist())

display(summary_df)

if "status" in summary_df.columns:
    print("\n=== 正常完成 ===")
    display(summary_df[summary_df["status"] == "OK"])

    print("\n=== SKIP / ERROR ===")
    display(summary_df[summary_df["status"] != "OK"])
else:
    print("\n⚠️ 结果文件中没有 status 列。")
    print("说明 R 端输出格式仍然不符合预期，请检查上一个 Cell 的 R 输出。")

结果文件： ..\results\de_edgeR\DE_summary_old_vs_young.tsv
结果维度： (10, 15)
结果列：
['tissue', 'cell_type', 'status', 'reason', 'formula', 'n_samples', 'n_young', 'n_old', 'residual_df', 'genes_before', 'genes_after_filterByExpr', 'genes_removed', 'significant_genes', 'up_genes', 'down_genes']


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,55.0,23.0,32.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,266.0,111.0,155.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,8,5.0,3.0,5.0,3000.0,416.0,2584.0,12.0,2.0,10.0
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,39.0,28.0,11.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1318.0,1682.0,0.0,0.0,0.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1440.0,1560.0,9.0,4.0,5.0
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== 正常完成 ===


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,55.0,23.0,32.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,266.0,111.0,155.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,8,5.0,3.0,5.0,3000.0,416.0,2584.0,12.0,2.0,10.0
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,39.0,28.0,11.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1318.0,1682.0,0.0,0.0,0.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,4,2.0,2.0,1.0,3000.0,1440.0,1560.0,9.0,4.0,5.0



=== SKIP / ERROR ===


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 最终分析标准

本周只采用：

```text
~ age_group + sex + technology
```

显著差异基因：

```text
FDR < 0.05
且
|logFC| >= 1
```

其中 `age_groupold` 是主要生物学检验。

**不会再生成连续年龄 `age_months` 的分析结果。**
